# Projet Assurance Assistance — Pipeline complet
## Nettoyage, Exploration & Analyses descriptives

**Contexte** : Ce notebook centralise l'intégralité du pipeline de traitement des données d'assistance automobile 2021-2022.
Il couvre le chargement des fichiers bruts, la détection et la correction des anomalies, puis les analyses descriptives et multivariées.

**Tables utilisées :**
| Table | Description | Volume estimé |
|---|---|---|
| `Dossiers` | Dossiers d'assistance ouverts en 2021-2022 | ~101 000 lignes |
| `Temps` | Temps passés par agent (matricule) sur chaque dossier | variable |
| `Ressources` | Caractéristiques des agents (CAC/CAS, contrat, lieu de travail) | variable |

**Plan :**
1. Configuration & imports
2. Fonctions de chargement
3. Fonctions de nettoyage
4. Chargement des données brutes
5. Exploration initiale
6. Nettoyage & tableau des anomalies
7. Analyses descriptives — Dossiers
8. Analyses descriptives — Temps
9. Analyses descriptives — Ressources
10. Analyses multivariées
11. Sauvegarde
12. Synthèse & Analyse des résultats
13. Mise en production & Partage

In [ ]:
import io, csv
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option('future.infer_string', False)  # compatibilité pandas 3.x — colonnes string en object
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
sns.set_theme(style='whitegrid', palette='muted')
COLORS = sns.color_palette('muted')
print('Imports OK')

---
## Section 1 — Configuration

Définition des chemins relatifs au projet et création automatique des dossiers nécessaires.
Aucun chemin absolu — fonctionne sur n'importe quel PC du groupe.

In [ ]:
PROJECT_ROOT = Path().resolve().parent  # projet_assurance/

DATA_DIR    = PROJECT_ROOT / 'data'
RAW_DIR     = PROJECT_ROOT / 'data' / 'raw'
REPORTS_DIR = PROJECT_ROOT / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'

DOSSIERS_CLEAN   = DATA_DIR / 'dossiers_clean.csv'
TEMPS_CLEAN      = DATA_DIR / 'temps_clean.csv'
RESSOURCES_CLEAN = DATA_DIR / 'ressources_clean.csv'

ANNEE_MIN, ANNEE_MAX = 2021, 2022

DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('Configuration OK — racine projet :', PROJECT_ROOT)

### Chargement des fichiers sources

Utilisez les boutons ci-dessous pour déposer vos **3 fichiers CSV bruts**.
Cliquez ensuite sur **Valider** pour les enregistrer dans le projet.

In [3]:
import ipywidgets as widgets
from IPython.display import display

upload_dos = widgets.FileUpload(description='Dossiers',   accept='.csv', multiple=False)
upload_tps = widgets.FileUpload(description='Temps',      accept='.csv', multiple=False)
upload_res = widgets.FileUpload(description='Ressources', accept='.csv', multiple=False)
btn        = widgets.Button(description='Valider les fichiers', button_style='success',
                            icon='check', layout=widgets.Layout(width='220px', height='35px'))
out = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<b>Fichier Dossiers :</b>'),   upload_dos,
    widgets.HTML('<b>Fichier Temps :</b>'),       upload_tps,
    widgets.HTML('<b>Fichier Ressources :</b>'),  upload_res,
    widgets.HBox([btn]), out
], layout=widgets.Layout(gap='6px')))

def _save_upload(upload, dest):
    val = upload.value
    if not val:
        return None
    try:
        item = list(val)[0]
        content, name = item['content'], item['name']
    except (TypeError, AttributeError):
        name, info = next(iter(val.items()))
        content = info['content']
    with open(dest, 'wb') as f:
        f.write(content)
    return name, len(content)

def on_valider(b):
    with out:
        out.clear_output()
        uploads = [
            (upload_dos, RAW_DIR / 'dossiers.csv',  'Dossiers'),
            (upload_tps, RAW_DIR / 'temps.csv',      'Temps'),
            (upload_res, RAW_DIR / 'ressources.csv', 'Ressources'),
        ]
        ok = True
        for upload, dest, label in uploads:
            result = _save_upload(upload, dest)
            if result:
                name, size = result
                print(f'  OK  {label} : {name}  ({size/1024:.0f} Ko)')
            else:
                print(f'  MANQUANT  {label} : aucun fichier fourni')
                ok = False
        if ok:
            print()
            print('Tous les fichiers sont prets. Vous pouvez continuer.')

btn.on_click(on_valider)

---
## Section 2 — Fonctions de chargement

Ces trois fonctions lisent les fichiers CSV bruts et retournent un DataFrame pandas propre.

In [ ]:
def _load_csv(path, min_cols):
    for enc in ('utf-8-sig', 'utf-8', 'latin-1'):
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            if len(df.columns) >= min_cols:
                df.columns = df.columns.str.strip()
                return df
        except Exception:
            continue
    raise ValueError(f'Impossible de lire {path}')


def load_dossiers(filepath=None):
    path = Path(filepath or DOSSIERS_FILE)
    if path.suffix == '.xlsx':
        raw = pd.read_excel(path, sheet_name=0, header=None)
        df = pd.read_csv(io.StringIO('\n'.join(raw[0].astype(str))))
    else:
        raw_bytes = open(path, 'rb').read().lstrip(b'\xef\xbb\xbf')
        inner_rows = [r[0] for r in csv.reader(io.StringIO(raw_bytes.decode('latin-1'))) if r]
        df = pd.read_csv(io.StringIO('\n'.join(inner_rows)), low_memory=False)
    df.columns = df.columns.str.replace(r'^[﻿"]+|["]+$', '', regex=True).str.strip()
    return df


def load_temps(filepath=None):
    return _load_csv(filepath or TEMPS_FILE, min_cols=5)


def load_ressources(filepath=None):
    return _load_csv(filepath or RESSOURCES_FILE, min_cols=8)


print('Fonctions de chargement définies')

---
## Section 3 — Fonctions de nettoyage

Ces fonctions prennent un DataFrame brut et retournent `(df_nettoyé, anomalies)`.
- `df_nettoyé` : DataFrame corrigé, prêt pour l'analyse
- `anomalies`  : liste de dicts décrivant chaque problème (Table, Colonne, Type, Nb lignes, Retraitement)

In [ ]:
def _anom(table, col, type_, n, retr):
    return {'Table': table, 'Colonne': col, 'Type': type_, 'Nb lignes': int(n), 'Retraitement': retr}


def clean_dossiers(df):
    anom, df = [], df.copy()

    # '???' → NaN
    n = (df == '???').sum().sum()
    if n:
        anom.append(_anom('Dossiers', 'Plusieurs', "Valeur '???'", n, 'Remplacement par NaN'))
        df.replace('???', np.nan, inplace=True)

    # date.ouverture : str.extract gère heure incluse + format invalide en un seul passage
    df['date.ouverture'] = pd.to_datetime(
        df['date.ouverture'].astype(str).str.extract(r'(\d{4}/\d{2}/\d{2})')[0],
        format='%Y/%m/%d', errors='coerce')
    n_inv = df['date.ouverture'].isna().sum()
    if n_inv:
        anom.append(_anom('Dossiers', 'date.ouverture', 'Format invalide (heure incluse ou valeur non reconnue)', n_inv, 'Mise à NaN'))
    mask = df['date.ouverture'].dt.year.notna() & ~df['date.ouverture'].dt.year.between(ANNEE_MIN, ANNEE_MAX)
    if mask.sum():
        anom.append(_anom('Dossiers', 'date.ouverture', 'Date hors 2021-2022', mask.sum(), 'Mise à NaN'))
        df.loc[mask, 'date.ouverture'] = np.nan

    # date.de.survenance
    df['date.de.survenance'] = pd.to_datetime(
        df['date.de.survenance'].astype(str).str.strip().str.extract(r'(\d{4}/\d{2}/\d{2})')[0],
        format='%Y/%m/%d', errors='coerce')
    n_surv = df['date.de.survenance'].isna().sum()
    if n_surv:
        anom.append(_anom('Dossiers', 'date.de.survenance', 'Format/valeur invalide', n_surv, 'Mise à NaN'))

    # Décalage de colonnes : shift(-1, axis=1) corrige le décalage en un appel
    SHIFT_COLS = ['Type.d.energie', 'Outil.d.assistance', 'Assistance.ou.Administratif',
                  'TOP.D.R', 'TOP.VR', 'TOP.Rappat.valide', 'TOP.Poursuite', 'TOP.Recup', 'TOP.Autres.Garanties']
    mask = df['Type.d.energie'].isin({'MCS', 'Higgins'})
    if mask.sum():
        anom.append(_anom('Dossiers', 'Type.d.energie/Outil/Assist.',
                          'Décalage colonnes (Distance_survenance présente)', mask.sum(), 'Corrigé par shift'))
        df.loc[mask, SHIFT_COLS] = df.loc[mask, SHIFT_COLS].shift(-1, axis=1).values

    # Date dans Cause.intervention → NaN
    mask = df['Cause.intervention'].astype(str).str.match(r'^\d{4}/\d{2}/\d{2}$')
    if mask.sum():
        anom.append(_anom('Dossiers', 'Cause.intervention', 'Date dans colonne cause', mask.sum(), 'Mise à NaN'))
        df.loc[mask, 'Cause.intervention'] = np.nan

    # Type.d.energie hors référentiel → NaN
    mask = ~df['Type.d.energie'].isin({'Diesel','Essence','Electricité','Hybride','Autre','GPL','inconnu'}) & df['Type.d.energie'].notna()
    if mask.sum():
        anom.append(_anom('Dossiers', 'Type.d.energie', 'Valeur hors référentiel', mask.sum(), 'Mise à NaN'))
        df.loc[mask, 'Type.d.energie'] = np.nan

    # Assistance.ou.Administratif hors référentiel → NaN
    mask = ~df['Assistance.ou.Administratif'].isin({'Assistance', 'Administratif'}) & df['Assistance.ou.Administratif'].notna()
    if mask.sum():
        anom.append(_anom('Dossiers', 'Assistance.ou.Administratif', 'Valeur hors référentiel', mask.sum(), 'Mise à NaN'))
        df.loc[mask, 'Assistance.ou.Administratif'] = np.nan

    # TOP colonnes → 0/1 Int64
    for col in ['TOP.D.R', 'TOP.VR', 'TOP.Rappat.valide', 'TOP.Poursuite', 'TOP.Recup']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        mask = ~df[col].isin([0, 1]) & df[col].notna()
        if mask.sum():
            anom.append(_anom('Dossiers', col, 'Valeur hors [0,1]', mask.sum(), 'Mise à NaN'))
            df.loc[mask, col] = np.nan
        df[col] = df[col].astype('Int64')
    df['TOP.Autres.Garanties'] = pd.to_numeric(df['TOP.Autres.Garanties'], errors='coerce').astype('Int64')

    # Valeurs manquantes résiduelles
    for col in ['Formule', 'Cause.intervention', 'Type.d.energie']:
        n = df[col].isna().sum()
        if n:
            anom.append(_anom('Dossiers', col, 'Valeurs manquantes', n, 'Conservées comme NaN'))

    # Matricule 171 — créations auto (absentes de la table Temps par construction)
    mask = df['Matricule.de.traitement'].astype(str).str.strip() == '171'
    anom.append(_anom('Dossiers', 'Matricule.de.traitement',
                      'Dossiers créés automatiquement (matricule 171)', mask.sum(),
                      'Flag flag_auto_creation = 1'))
    df['flag_auto_creation'] = mask.astype(int)

    # Apprentissage_Test — dossiers TEST sans temps
    if 'Apprentissage_Test' in df.columns:
        mask = df['Apprentissage_Test'].astype(str).str.strip() == 'Test'
        anom.append(_anom('Dossiers', 'Apprentissage_Test',
                          'Dossiers TEST (duree.corrigee vide, ~11%)', mask.sum(), 'Flag flag_test = 1'))
        df['flag_test'] = mask.astype(int)

    # Distance.de.survenance négative → NaN
    if 'Distance.de.survenance' in df.columns:
        df['Distance.de.survenance'] = pd.to_numeric(df['Distance.de.survenance'], errors='coerce')
        mask = df['Distance.de.survenance'] < 0
        if mask.sum():
            anom.append(_anom('Dossiers', 'Distance.de.survenance', 'Distance négative', mask.sum(), 'Mise à NaN'))
            df.loc[mask, 'Distance.de.survenance'] = np.nan

    return df, anom


def clean_temps(df):
    anom, df = [], df.copy()

    mask = df['duree.corrigee'] <= 0
    if mask.sum():
        anom.append(_anom('Temps', 'duree.corrigee', 'Durée ≤ 0', mask.sum(), 'Mise à NaN'))
        df.loc[mask, 'duree.corrigee'] = np.nan

    mask_ext = df['duree.corrigee'] > 28800
    if mask_ext.sum():
        anom.append(_anom('Temps', 'duree.corrigee', 'Durée > 8h (28 800 sec)', mask_ext.sum(), 'Conservées avec flag'))
    df['flag_duree_extreme'] = mask_ext.astype(int)

    df['Date.debut.traitement'] = pd.to_datetime(df['Date.debut.traitement'], dayfirst=True, errors='coerce')

    for col, n in df.isna().sum().items():
        if n:
            anom.append(_anom('Temps', col, 'Valeurs manquantes', n, 'Conservées comme NaN'))

    return df, anom


def clean_ressources(df):
    anom, df = [], df.copy()

    df['Date.presence'] = pd.to_datetime(df['Date.presence'], dayfirst=True, errors='coerce')
    n = df['Date.presence'].isna().sum()
    if n:
        anom.append(_anom('Ressources', 'Date.presence', 'Format date invalide', n, 'Mise à NaT'))

    mask = df['Duree.travail'] > 24
    if mask.sum():
        anom.append(_anom('Ressources', 'Duree.travail', 'Durée > 24h (impossible)', mask.sum(), 'Mise à NaN'))
        df.loc[mask, 'Duree.travail'] = np.nan

    mask = ~df['Temps.travail'].isin({30, 50, 70, 80, 100}) & df['Temps.travail'].notna()
    if mask.sum():
        anom.append(_anom('Ressources', 'Temps.travail', 'Hors {30,50,70,80,100}', mask.sum(), 'Conservées — à documenter'))

    mask = ~df['Type.de.contrat'].isin({'CDI', 'CDD', 'CDS'}) & df['Type.de.contrat'].notna()
    if mask.sum():
        anom.append(_anom('Ressources', 'Type.de.contrat', 'Hors CDI/CDD/CDS', mask.sum(), 'Mise à NaN'))
        df.loc[mask, 'Type.de.contrat'] = np.nan

    mask = df['Experience'] < 0
    if mask.sum():
        anom.append(_anom('Ressources', 'Experience', 'Expérience négative', mask.sum(), 'Mise à NaN'))
        df.loc[mask, 'Experience'] = np.nan

    return df, anom


def build_anomaly_table(all_anomalies):
    return pd.DataFrame(all_anomalies)


print('Fonctions de nettoyage définies')

---
## Section 4 — Chargement des données brutes

On charge les trois tables depuis les fichiers sources sans aucune transformation.
L'objectif est d'avoir une photographie fidèle des données telles qu'elles arrivent.

In [6]:
DOSSIERS_FILE   = RAW_DIR / 'dossiers.csv'
TEMPS_FILE      = RAW_DIR / 'temps.csv'
RESSOURCES_FILE = RAW_DIR / 'ressources.csv'

print('Chargement Dossiers...')
df_dos_raw = load_dossiers(DOSSIERS_FILE)
print(f'  -> {len(df_dos_raw):,} lignes, {len(df_dos_raw.columns)} colonnes')

print('Chargement Temps...')
df_tps_raw = load_temps(TEMPS_FILE)
print(f'  -> {len(df_tps_raw):,} lignes, {len(df_tps_raw.columns)} colonnes')

print('Chargement Ressources...')
df_res_raw = load_ressources(RESSOURCES_FILE)
print(f'  -> {len(df_res_raw):,} lignes, {len(df_res_raw.columns)} colonnes')

Chargement Dossiers...
  -> 101,206 lignes, 17 colonnes
Chargement Temps...
  -> 431,598 lignes, 5 colonnes
Chargement Ressources...
  -> 389,353 lignes, 9 colonnes


---
## Section 5 — Exploration initiale

Avant de nettoyer, on inspecte la structure de chaque table : types de colonnes, valeurs manquantes et premières lignes.
Cette étape permet de valider que le chargement s'est bien passé et d'avoir une vue d'ensemble des problèmes à traiter.

### 5.1 Table Dossiers

In [7]:
print('=== Types de colonnes — Dossiers ===')
print(df_dos_raw.dtypes)
df_dos_raw.head(5)

=== Types de colonnes — Dossiers ===
Numero_dossier_ID               object
Client                          object
Formule                         object
date.ouverture                  object
heure.ouverture                 object
Matricule.de.traitement         object
Cause.intervention              object
date.de.survenance              object
Type.d.energie                  object
Outil.d.assistance              object
Assistance.ou.Administratif     object
TOP.D.R                          int64
TOP.VR                           int64
TOP.Rappat.valide                int64
TOP.Poursuite                    int64
TOP.Recup                        int64
TOP.Autres.Garanties           float64
dtype: object


,Numero_dossier_ID,Client,Formule,date.ouverture,heure.ouverture,Matricule.de.traitement,Cause.intervention,date.de.survenance,Type.d.energie,Outil.d.assistance,Assistance.ou.Administratif,TOP.D.R,TOP.VR,TOP.Rappat.valide,TOP.Poursuite,TOP.Recup,TOP.Autres.Garanties
0,7494402,C5,F99,2021/02/11,10:38:00,326,Panne mécanique,2021/02/11,Diesel,MCS,Assistance,1,0,0,0,0,0.0
1,7569082,C4,F14,2022/07/12,18:40:00,1164,Panne mécanique,2022/07/12,Essence,MCS,???,1,0,0,0,0,0.0
2,8111190,C0,F1,2021/01/08,12:32:00,1337,Panne mécanique,2021/01/08,Essence,MCS,Assistance,1,1,0,0,0,0.0
3,5630809,C7,F72,2021/11/08,19:46:00,1322,Panne mécanique,2021/11/08,Essence,MCS,Assistance,1,0,0,0,0,0.0
4,8179409,C4,F4,2021/08/11,18:12:00,2306,Panne mécanique,2021/08/11,Essence,Higgins,???,1,0,0,0,0,0.0


In [8]:
print('=== Valeurs manquantes — Dossiers ===')
missing = df_dos_raw.isnull().sum()
missing_pct = (missing / len(df_dos_raw) * 100).round(2)
pd.DataFrame({'Nb manquants': missing, '% manquants': missing_pct})[missing > 0]

=== Valeurs manquantes — Dossiers ===


,Nb manquants,% manquants
Formule,949,0.94
Cause.intervention,999,0.99
Type.d.energie,994,0.98
TOP.Autres.Garanties,28,0.03


### 5.2 Table Temps

In [9]:
print('=== Types de colonnes — Temps ===')
print(df_tps_raw.dtypes)
print()
print(df_tps_raw.describe().round(2))
df_tps_raw.head(3)

=== Types de colonnes — Temps ===
Numero.dossier              int64
Matricule                   int64
Date.debut.traitement      object
heure.debut.traitement     object
duree.corrigee            float64
dtype: object

       Numero.dossier  Matricule  duree.corrigee
count       431598.00  431598.00       384127.00
mean       6997541.93    1039.17          312.31
std         887011.36     621.00          581.61
min        5465153.00     123.00            5.00
25%        6229503.00     519.00           52.00
50%        6990646.00     953.00          188.00
75%        7768250.00    1469.00          411.00
max        8536053.00    2662.00       253014.00


,Numero.dossier,Matricule,Date.debut.traitement,heure.debut.traitement,duree.corrigee
0,5465153,1979,2021/07/03,20:04,217.0
1,5465153,1323,2021/07/07,12:29,690.0
2,5465153,1735,2021/07/07,14:31,297.0


### 5.3 Table Ressources

In [10]:
print('=== Types de colonnes — Ressources ===')
print(df_res_raw.dtypes)
print()
print(df_res_raw.describe().round(2))
df_res_raw.head(3)

=== Types de colonnes — Ressources ===
Matricule            int64
Date.presence       object
Lieu.travail        object
Population          object
Site                object
Type.de.contrat     object
Duree.travail      float64
Temps.travail        int64
Experience           int64
dtype: object

       Matricule  Duree.travail  Temps.travail  Experience
count   389353.0      389353.00      389353.00   389353.00
mean      1030.3           6.15          72.08     1306.51
std        591.5           1.72           9.67     1428.08
min        123.0           0.02          30.00        0.00
25%        542.0           5.50          70.00      174.00
50%        967.0           7.33          70.00      579.00
75%       1450.0           7.33          80.00     2388.00
max       3192.0          23.42         100.00     6186.00


,Matricule,Date.presence,Lieu.travail,Population,Site,Type.de.contrat,Duree.travail,Temps.travail,Experience
0,1243,05/01/2021,TELE,CAS,A,CDI,7.333333,100,5783
1,1243,07/01/2021,TELE,CAS,A,CDI,0.883333,100,5784
2,1243,08/01/2021,TELE,CAS,A,CDI,7.333333,100,5785


---
## Section 6 — Nettoyage des données

On applique les fonctions de nettoyage définies en Section 3.
Chaque anomalie détectée est enregistrée avec son type, le nombre de lignes concernées et le retraitement appliqué.

In [11]:
df_dos, anom_dos = clean_dossiers(df_dos_raw)
df_tps, anom_tps = clean_temps(df_tps_raw)
df_res, anom_res = clean_ressources(df_res_raw)

print(f'Dossiers nettoyés   : {len(df_dos):,} lignes')
print(f'Temps nettoyés      : {len(df_tps):,} lignes')
print(f'Ressources nettoyées: {len(df_res):,} lignes')

Dossiers nettoyés   : 101,206 lignes
Temps nettoyés      : 431,598 lignes
Ressources nettoyées: 389,353 lignes


### 6.1 Tableau de synthèse des anomalies

Ce tableau récapitule l'ensemble des anomalies détectées dans les trois tables, classées par type de problème.
Il constitue la **traçabilité du nettoyage** et peut être joint au rapport final.

In [ ]:
tableau_anomalies = build_anomaly_table(anom_dos + anom_tps + anom_res)
tableau_anomalies.to_csv(REPORTS_DIR / 'tableau_anomalies.csv', index=False, encoding='utf-8-sig')
print(f'{len(tableau_anomalies)} types d anomalies détectés — sauvegardé dans reports/tableau_anomalies.csv')
tableau_anomalies

### 6.2 Contrôles de cohérence post-nettoyage

On vérifie trois règles métier fondamentales :
1. La date d'ouverture d'un dossier ne doit pas être antérieure à la date du sinistre.
2. Tout dossier présent dans la table Temps doit exister dans la table Dossiers.
3. Tout matricule dans la table Temps doit être référencé dans la table Ressources.

In [ ]:
incoherence = (df_dos['date.ouverture'] < df_dos['date.de.survenance']).sum()
print(f'Dossiers où ouverture < survenance (incohérent) : {incoherence}')

ids_dos = set(df_dos['Numero_dossier_ID'].astype(str))
ids_tps = set(df_tps['Numero.dossier'].astype(str))
print(f'Dossiers dans Temps absents de Dossiers : {len(ids_tps - ids_dos)}')

mats_res = set(df_res['Matricule'].astype(str))
mats_tps = set(df_tps['Matricule'].astype(str))
print(f'Matricules dans Temps absents de Ressources : {len(mats_tps - mats_res)}')

n_sans_temps = len(ids_dos - ids_tps)
n_auto = df_dos['flag_auto_creation'].sum()
print(f'\nDossiers de la table Dossiers sans ligne dans Temps : {n_sans_temps}')
print(f'  dont créés automatiquement (matricule système 171) : {n_auto}')
if 'flag_test' in df_dos.columns:
    n_test = df_dos['flag_test'].sum()
    print(f'  dont dossiers TEST (temps mis à vide)               : {n_test}')
    print(f'  Sous-total expliqué                                  : {n_auto + n_test}')
    print(f'  Résidu inexpliqué                                    : {n_sans_temps - n_auto - n_test}')

### 6.3 Sauvegarde des données nettoyées

In [14]:
df_dos.to_csv(DOSSIERS_CLEAN, index=False, encoding='utf-8-sig')
df_tps.to_csv(TEMPS_CLEAN,    index=False, encoding='utf-8-sig')
df_res.to_csv(RESSOURCES_CLEAN, index=False, encoding='utf-8-sig')
print('Données nettoyées sauvegardées dans data/')

Données nettoyées sauvegardées dans data/


---
## Section 7 — Analyses descriptives — Table Dossiers

Cette section explore la distribution des principales variables de la table Dossiers :
répartition par client, causes d'intervention, évolution temporelle, type d'énergie et services activés.

### 7.1 Répartition des dossiers par client (assureur)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df_dos['Client'].value_counts().plot(kind='bar', ax=ax, color=COLORS[0], edgecolor='white')
ax.set_title('Nombre de dossiers par client (assureur)', fontsize=13)
ax.set_xlabel('Client')
ax.set_ylabel('Nb dossiers')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_dossiers_par_client.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture du graphique :** Chaque barre représente un assureur partenaire.
Le client dominant concentre la majorité des dossiers — il constitue le cœur du portefeuille d'activité.
Un déséquilibre important entre clients peut indiquer une dépendance forte à un partenaire unique.

### 7.2 Causes d'intervention

In [ ]:
cause_counts = df_dos['Cause.intervention'].value_counts(dropna=True)
fig, ax = plt.subplots(figsize=(10, 5))
cause_counts.plot(kind='barh', ax=ax, color=COLORS[1], edgecolor='white')
ax.set_title("Répartition des causes d'intervention", fontsize=13)
ax.set_xlabel('Nb dossiers')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_causes_intervention.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture du graphique :** Les barres sont triées par fréquence décroissante.
Les causes les plus fréquentes (panne, accident, crevaison...) représentent les typologies de sinistres les plus courantes.
Cette répartition guide la priorisation des ressources et l'organisation des équipes d'assistance.

### 7.3 Évolution mensuelle du nombre de dossiers ouverts

In [ ]:
df_dos['mois_ouverture'] = df_dos['date.ouverture'].dt.to_period('M')
monthly = df_dos.groupby('mois_ouverture').size()

fig, ax = plt.subplots(figsize=(14, 4))
monthly.plot(ax=ax, marker='o', color=COLORS[2], linewidth=2)
ax.set_title('Évolution mensuelle du nombre de dossiers ouverts (2021-2022)', fontsize=13)
ax.set_xlabel('Mois')
ax.set_ylabel('Nb dossiers')
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_evolution_mensuelle.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture du graphique :** La courbe révèle la saisonnalité de l'activité sur 2 ans.
Des pics en hiver (grand froid, verglas) et en été (départs en vacances, longs trajets) sont typiques dans l'assistance automobile.
Les creux peuvent correspondre à des périodes de faible activité ou à des problèmes de saisie.

### 7.4 Répartition par type d'énergie du véhicule

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df_dos['Type.d.energie'].value_counts(dropna=True).plot(
    kind='pie', ax=ax, autopct='%1.1f%%', startangle=140,
    colors=sns.color_palette('pastel'))
ax.set_ylabel('')
ax.set_title("Répartition par type d'énergie du véhicule", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_type_energie.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture du graphique :** La domination du Diesel et de l'Essence reflète le parc automobile français 2021-2022.
La part des véhicules Électriques et Hybrides, encore minoritaire, est à surveiller sur les années suivantes
car leurs pannes ont des causes et des temps de traitement différents.

### 7.5 Taux d'activation des services d'assistance

In [ ]:
top_cols   = ['TOP.D.R', 'TOP.VR', 'TOP.Rappat.valide', 'TOP.Poursuite', 'TOP.Recup', 'TOP.Autres.Garanties']
top_labels = ['Dépannage/Remorquage', 'Véhicule Remplacement', 'Rapatriement',
              'Poursuite voyage', 'Récupération véhicule', 'Autres garanties']
taux = [df_dos[c].mean() * 100 for c in top_cols]

pd.DataFrame({'Service': top_labels, 'Taux (%)': [round(t, 1) for t in taux]}).to_csv(
    REPORTS_DIR / 'taux_activation_services.csv', index=False, encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(top_labels, taux, color=COLORS[3], edgecolor='white')
ax.set_xlabel('% de dossiers activant ce service')
ax.set_title("Taux d'activation des services d'assistance", fontsize=13)
for i, v in enumerate(taux):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_taux_services.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture du graphique :** Le Dépannage/Remorquage est le service le plus activé — c'est le cœur de métier de l'assistance.
Le Véhicule de Remplacement et le Rapatriement sont des services coûteux, donc leur taux d'activation impacte directement la rentabilité.
Les services peu activés (Poursuite voyage, Récupération) sont des garanties complémentaires moins sollicitées.

---
## Section 8 — Analyses descriptives — Table Temps

Cette section analyse la distribution des durées de traitement par dossier.
La durée de traitement est un indicateur clé de la charge de travail des agents.

### 8.1 Distribution du temps total de traitement par dossier

In [ ]:
tps_par_dos = df_tps.groupby('Numero.dossier')['duree.corrigee'].sum().reset_index()
tps_par_dos.columns = ['Numero_dossier_ID', 'temps_total_sec']
tps_par_dos['temps_total_min'] = tps_par_dos['temps_total_sec'] / 60

print('Statistiques descriptives du temps total par dossier (minutes) :')
print(tps_par_dos['temps_total_min'].describe().round(1))

tps_par_dos.to_csv(REPORTS_DIR / 'temps_par_dossier.csv', index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Distribution du temps total de traitement par dossier', fontsize=13)

q99 = tps_par_dos['temps_total_min'].quantile(0.99)
tps_par_dos[tps_par_dos['temps_total_min'] <= q99]['temps_total_min'].hist(
    bins=50, ax=axes[0], color=COLORS[0], edgecolor='white')
axes[0].set_title('Histogramme (tronqué au 99e percentile)', fontsize=11)
axes[0].set_xlabel('Minutes')
axes[0].set_ylabel('Nb dossiers')

axes[1].boxplot(tps_par_dos['temps_total_min'].dropna(), vert=False, patch_artist=True,
                boxprops=dict(facecolor=COLORS[1]))
axes[1].set_title('Boxplot (valeurs extrêmes visibles)', fontsize=11)
axes[1].set_xlabel('Minutes')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_distribution_temps.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture des graphiques :**
- L'**histogramme** montre la distribution des durées en excluant le 1% de valeurs les plus extrêmes pour une meilleure lisibilité. Une distribution asymétrique à droite est typique : la majorité des dossiers est traitée rapidement, mais quelques cas complexes prennent beaucoup plus de temps.
- Le **boxplot** révèle la médiane (trait central), les quartiles (boîte) et les valeurs aberrantes (points à droite). Les outliers représentent des dossiers exceptionnellement longs, à investiguer.

---
## Section 9 — Analyses descriptives — Table Ressources

Cette section décrit la composition des équipes : population (CAC/CAS), type de contrat et lieu de travail.
Ces informations sont essentielles pour comprendre les capacités opérationnelles.

### 9.1 Composition des équipes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Composition des équipes — Analyses univariées', fontsize=13, y=1.02)

agents = df_res.drop_duplicates('Matricule')
agents['Population'].value_counts().plot(kind='bar', ax=axes[0], color=COLORS[0], edgecolor='white')
axes[0].set_title('Population : CAC vs CAS', fontsize=11)
axes[0].set_xlabel('')
plt.setp(axes[0].get_xticklabels(), rotation=0)

agents['Type.de.contrat'].value_counts().plot(kind='bar', ax=axes[1], color=COLORS[1], edgecolor='white')
axes[1].set_title('Type de contrat', fontsize=11)
plt.setp(axes[1].get_xticklabels(), rotation=0)

df_res['Lieu.travail'].value_counts().plot(kind='bar', ax=axes[2], color=COLORS[2], edgecolor='white')
axes[2].set_title('Lieu de travail (TELE vs SITE)', fontsize=11)
plt.setp(axes[2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_composition_equipes.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture des graphiques :**
- **CAC vs CAS** : les CAC (Conseillers Assistance Clientèle) et CAS (Conseillers Assistance Sinistre) ont des rôles distincts. Un déséquilibre entre les deux peut indiquer un besoin de recrutement ciblé.
- **Type de contrat** : la proportion CDI/CDD/CDS reflète la stabilité des effectifs. Une forte proportion de CDD peut expliquer une variabilité dans les performances.
- **Lieu de travail** : la répartition TELE/SITE renseigne sur l'adoption du télétravail, un facteur clé depuis 2020.

---
## Section 10 — Analyses multivariées

On joint les trois tables pour croiser les informations et répondre à des questions plus riches :
- Quelles causes d'intervention prennent le plus de temps ?
- Les dossiers d'assistance et administratifs mobilisent-ils les mêmes services ?
- Comment évolue l'usage des outils MCS et Higgins dans le temps ?
- Quelles variables sont corrélées entre elles ?

### 10.1 Jointure des tables

In [ ]:
tps_grp = df_tps.groupby('Numero.dossier').agg(
    temps_total_sec=('duree.corrigee', 'sum'),
    nb_intervenants=('Matricule', 'nunique')
).reset_index().rename(columns={'Numero.dossier': 'Numero_dossier_ID'})

df_dos['Numero_dossier_ID'] = df_dos['Numero_dossier_ID'].astype(str)
tps_grp['Numero_dossier_ID'] = tps_grp['Numero_dossier_ID'].astype(str)

df_joint = df_dos.merge(tps_grp, on='Numero_dossier_ID', how='left')
df_joint['temps_total_min']    = df_joint['temps_total_sec'] / 60
df_joint['Apprentissage_Test'] = np.where(
    df_joint['temps_total_min'].isna() | (df_joint['temps_total_min'] == 0),
    'Test', 'Apprentissage'
)

print(f'Table jointe : {len(df_joint):,} lignes')
print(f'Apprentissage : {(df_joint["Apprentissage_Test"] == "Apprentissage").sum():,} ({(df_joint["Apprentissage_Test"] == "Apprentissage").mean()*100:.0f}%)')
print(f'Test          : {(df_joint["Apprentissage_Test"] == "Test").sum():,} ({(df_joint["Apprentissage_Test"] == "Test").mean()*100:.0f}%)')
df_joint.head(3)

### 10.2 Temps moyen de traitement par cause d'intervention

In [ ]:
tps_cause = df_joint.groupby('Cause.intervention')['temps_total_min'].agg(['mean', 'median', 'count'])
tps_cause = tps_cause[tps_cause['count'] > 100].sort_values('mean')

fig, ax = plt.subplots(figsize=(10, 5))
tps_cause['mean'].plot(kind='barh', ax=ax, color=COLORS[0], edgecolor='white', label='Moyenne')
tps_cause['median'].plot(kind='barh', ax=ax, color=COLORS[1], edgecolor='white', alpha=0.6, label='Médiane')
ax.set_title("Temps moyen et médian de traitement par cause d'intervention", fontsize=13)
ax.set_xlabel('Minutes')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / '08_temps_par_cause.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture du graphique :** La superposition de la moyenne et de la médiane est révélatrice :
- Si **moyenne >> médiane** : des cas très longs tirent la moyenne vers le haut (distribution asymétrique).
- Les causes avec les temps les plus élevés nécessitent davantage de ressources spécialisées.
- Ce graphique peut servir de base pour définir des **standards de temps** par type de sinistre.

### 10.3 Services activés : Assistance vs Administratif

In [ ]:
top_cols = ['TOP.D.R', 'TOP.VR', 'TOP.Rappat.valide', 'TOP.Poursuite', 'TOP.Recup']
grp = df_joint.groupby('Assistance.ou.Administratif')[top_cols].mean() * 100
grp.T.plot(kind='bar', figsize=(10, 5), edgecolor='white')
plt.title("Taux d'activation des services : Assistance vs Administratif", fontsize=13)
plt.ylabel('% de dossiers activant le service')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Type de dossier')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '09_services_assist_vs_admin.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture du graphique :** Les dossiers d'Assistance (intervention terrain) activent davantage de services physiques
(Dépannage, Rapatriement) tandis que les dossiers Administratifs se concentrent sur des garanties sans déplacement.
Un écart important entre les deux types confirme qu'ils mobilisent des ressources et des compétences différentes.

### 10.4 Évolution mensuelle par outil (MCS vs Higgins)

In [ ]:
df_joint['mois'] = df_joint['date.ouverture'].dt.to_period('M')
outil_mois = df_joint.groupby(['mois', 'Outil.d.assistance']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 4))
outil_mois.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Évolution mensuelle du nombre de dossiers par outil (MCS vs Higgins)', fontsize=13)
ax.set_xlabel('Mois')
ax.set_ylabel('Nb dossiers')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '10_evolution_outils.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture du graphique :** Ce graphique illustre la **migration progressive** de l'outil MCS vers Higgins sur la période 2021-2022.
La courbe Higgins monte au fur et à mesure que MCS décline, confirmant le déploiement progressif du nouvel outil.
Un chevauchement des deux courbes correspond à la période de coexistence des deux systèmes.

### 10.5 Matrice de corrélation entre variables numériques

In [ ]:
num_cols = [c for c in ['temps_total_min', 'nb_intervenants', 'TOP.D.R', 'TOP.VR',
                         'TOP.Rappat.valide', 'TOP.Poursuite', 'TOP.Recup', 'TOP.Autres.Garanties']
            if c in df_joint.columns]
corr = df_joint[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax,
            linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Matrice de corrélation — Variables numériques', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '11_matrice_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture de la heatmap :**
- Les valeurs proches de **+1** (rouge foncé) indiquent une forte corrélation positive : les deux variables augmentent ensemble.
- Les valeurs proches de **-1** (bleu foncé) indiquent une corrélation négative : quand l'une augmente, l'autre diminue.
- Les valeurs proches de **0** (blanc) indiquent l'absence de corrélation linéaire.
- Une corrélation élevée entre `temps_total_min` et `nb_intervenants` confirmerait que les dossiers complexes mobilisent plus d'agents et durent plus longtemps.

### 10.6 Expérience moyenne des agents selon le lieu de travail

In [ ]:
exp_lieu = df_res.groupby(['Lieu.travail', 'Population'])['Experience'].mean().unstack()
print('Expérience moyenne (jours) selon le lieu de travail et la population :')
print(exp_lieu.round(0))

fig, ax = plt.subplots(figsize=(7, 4))
exp_lieu.plot(kind='bar', ax=ax, edgecolor='white')
ax.set_title("Expérience moyenne (jours) : Télétravail vs Présentiel", fontsize=13)
ax.set_ylabel("Jours d'expérience")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '12_experience_par_lieu.png', dpi=150, bbox_inches='tight')
plt.show()

**Lecture du graphique :** Ce graphique croise le lieu de travail (TELE/SITE) avec la population (CAC/CAS).
Si les agents les plus expérimentés tendent à travailler davantage en télétravail, cela peut indiquer une politique
de confiance accordée aux profils seniors. À l'inverse, si les juniors sont plus en présentiel, cela reflète un besoin d'encadrement.

---
## Section 11 — Sauvegarde de la table enrichie

On exporte la table jointe (Dossiers + agrégats Temps) pour les analyses futures ou l'alimentation du dashboard.

In [28]:
df_joint.to_csv(DATA_DIR / 'dossiers_enrichis.csv', index=False, encoding='utf-8-sig')
print('Table enrichie sauvegardée → data/dossiers_enrichis.csv')

Table enrichie sauvegardée → data/dossiers_enrichis.csv


---
## Section 12 — Synthèse & Analyse des résultats

Cette section consolide les principaux enseignements tirés des analyses descriptives et multivariées.
Elle structure les résultats en trois axes : qualité des données, comportement de l'activité, et profil des ressources.

### 12.1 Bilan qualité des données

Avant d'interpréter les résultats, il est important de connaître l'état des données après nettoyage.

In [ ]:
cols_cles = ['date.ouverture', 'date.de.survenance', 'Client', 'Formule',
             'Cause.intervention', 'Type.d.energie', 'Outil.d.assistance',
             'Assistance.ou.Administratif']

notna = df_dos[cols_cles].notna().sum()
bilan = pd.DataFrame({
    'Nb renseignés': notna.values,
    'Nb manquants':  (len(df_dos) - notna).values,
    '% complétude':  (notna / len(df_dos) * 100).round(1).values,
}, index=cols_cles)

bilan.to_csv(REPORTS_DIR / 'bilan_completude.csv', encoding='utf-8-sig')

print('=== Taux de complétude des colonnes clés après nettoyage ===')
bilan

**Interprétation :** Un taux de complétude élevé (> 95 %) valide la fiabilité de la colonne pour les analyses.
Les colonnes avec un taux faible (ex. `Cause.intervention`, `Type.d.energie`) doivent être interprétées avec prudence :
les résultats portent sur les lignes renseignées uniquement et peuvent ne pas être représentatifs de l'ensemble.

### 12.2 Principaux enseignements — Activité

Résumé chiffré des indicateurs clés issus des analyses.

In [ ]:
total_dossiers    = len(df_dos)
pct_assistance    = (df_dos['Assistance.ou.Administratif'] == 'Assistance').mean() * 100
pct_admin         = (df_dos['Assistance.ou.Administratif'] == 'Administratif').mean() * 100
taux_dr           = df_dos['TOP.D.R'].mean() * 100
taux_vr           = df_dos['TOP.VR'].mean() * 100
taux_rapat        = df_dos['TOP.Rappat.valide'].mean() * 100
tps_median_min    = df_joint['temps_total_min'].median()
nb_interv_median  = df_joint['nb_intervenants'].median()

pd.DataFrame([
    {'Indicateur': 'Nombre total de dossiers',          'Valeur': total_dossiers,              'Unité': 'dossiers'},
    {'Indicateur': 'Part dossiers Assistance',          'Valeur': round(pct_assistance, 1),    'Unité': '%'},
    {'Indicateur': 'Part dossiers Administratif',       'Valeur': round(pct_admin, 1),         'Unité': '%'},
    {'Indicateur': 'Taux activation Dépannage/Remor.',  'Valeur': round(taux_dr, 1),           'Unité': '%'},
    {'Indicateur': 'Taux activation Véh. Remplacement', 'Valeur': round(taux_vr, 1),           'Unité': '%'},
    {'Indicateur': 'Taux activation Rapatriement',      'Valeur': round(taux_rapat, 1),        'Unité': '%'},
    {'Indicateur': 'Temps médian de traitement',        'Valeur': round(tps_median_min, 1),    'Unité': 'min'},
    {'Indicateur': 'Nb médian intervenants/dossier',    'Valeur': nb_interv_median,            'Unité': 'agents'},
]).to_csv(REPORTS_DIR / 'indicateurs_activite.csv', index=False, encoding='utf-8-sig')

print('╔══════════════════════════════════════════════════════╗')
print('║         INDICATEURS CLÉS — ACTIVITÉ 2021-2022       ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Nombre total de dossiers         : {total_dossiers:>10,}     ║')
print(f'║  Part dossiers Assistance         : {pct_assistance:>9.1f} %    ║')
print(f'║  Part dossiers Administratif      : {pct_admin:>9.1f} %    ║')
print(f'║  Taux activation Dépannage/Remor. : {taux_dr:>9.1f} %    ║')
print(f'║  Taux activation Véh. Remplacement: {taux_vr:>9.1f} %    ║')
print(f'║  Taux activation Rapatriement     : {taux_rapat:>9.1f} %    ║')
print(f'║  Temps médian de traitement       : {tps_median_min:>9.1f} min  ║')
print(f'║  Nb médian d intervenants/dossier : {nb_interv_median:>9.1f}      ║')
print('╚══════════════════════════════════════════════════════╝')

**Interprétation :**
- La majorité des dossiers sont des dossiers d'**Assistance** (intervention terrain), ce qui confirme le cœur de métier.
- Le **Dépannage/Remorquage** est le service le plus sollicité : il représente la prestation de base systématiquement déclenchée.
- Le **Véhicule de Remplacement** et le **Rapatriement**, plus coûteux, sont activés sur une fraction des dossiers — leur taux est un levier important pour la maîtrise des coûts.
- Le **temps médian de traitement** reflète la charge réelle par dossier. Un écart important avec la moyenne signalerait des cas atypiques à surveiller.

### 12.3 Principaux enseignements — Ressources humaines

In [ ]:
nb_agents = df_res['Matricule'].nunique()
pct_tele  = (df_res['Lieu.travail'] == 'TELE').mean() * 100
pct_site  = (df_res['Lieu.travail'] == 'SITE').mean() * 100
agents    = df_res.drop_duplicates('Matricule')
exp_moy   = agents['Experience'].mean()
pct_cdi   = (agents['Type.de.contrat'] == 'CDI').mean() * 100

pd.DataFrame([
    {'Indicateur': 'Nombre d agents uniques',          'Valeur': nb_agents,          'Unité': 'agents'},
    {'Indicateur': 'Jours de présence en télétravail', 'Valeur': round(pct_tele, 1), 'Unité': '%'},
    {'Indicateur': 'Jours de présence en présentiel',  'Valeur': round(pct_site, 1), 'Unité': '%'},
    {'Indicateur': 'Expérience moyenne',               'Valeur': round(exp_moy, 1),  'Unité': 'jours'},
    {'Indicateur': 'Part des agents en CDI',           'Valeur': round(pct_cdi, 1),  'Unité': '%'},
]).to_csv(REPORTS_DIR / 'indicateurs_rh.csv', index=False, encoding='utf-8-sig')

print('╔══════════════════════════════════════════════════════╗')
print('║         INDICATEURS CLÉS — RESSOURCES HUMAINES      ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Nombre d agents uniques          : {nb_agents:>10,}     ║')
print(f'║  Jours de présence en télétravail : {pct_tele:>9.1f} %    ║')
print(f'║  Jours de présence en présentiel  : {pct_site:>9.1f} %    ║')
print(f'║  Expérience moyenne (jours)       : {exp_moy:>9.1f}      ║')
print(f'║  Part des agents en CDI           : {pct_cdi:>9.1f} %    ║')
print('╚══════════════════════════════════════════════════════╝')

**Interprétation :**
- La répartition TELE/SITE renseigne sur l'organisation du travail : un fort taux de télétravail indique une bonne maturité des outils à distance (Higgins notamment).
- L'**expérience moyenne** des agents en jours est un proxy de la séniorité des équipes. Elle influence directement la qualité et la rapidité du traitement.
- La **part des CDI** reflète la stabilité des effectifs : un taux élevé favorise la rétention des savoir-faire.

### 12.4 Points d'attention et limites

Avant toute prise de décision basée sur ces analyses, il convient de noter les limites suivantes.

| Point d'attention | Impact potentiel |
|---|---|
| Données manquantes sur `Cause.intervention` | Les analyses par cause portent sur un sous-ensemble — résultats à pondérer |
| Décalage de colonnes corrigé heuristiquement | Les lignes corrigées peuvent contenir des erreurs résiduelles |
| Durées > 8h flaggées mais conservées | Les statistiques de temps incluent des valeurs potentiellement erronées |
| Période limitée à 2021-2022 | Les tendances identifiées ne sont pas forcément représentatives d'autres années |
| Jointure Temps → Dossiers non exhaustive | Les dossiers sans temps enregistré sont exclus des analyses de durée |

---
## Section 13 — Mise en production & Partage

Cette section décrit comment utiliser et diffuser les résultats produits par ce pipeline.

### 13.1 Fichiers produits

Le pipeline génère les fichiers suivants, prêts à être utilisés en aval.

In [ ]:
fichiers = {
    'data/dossiers_clean.csv'              : 'Table Dossiers nettoyée',
    'data/temps_clean.csv'                 : 'Table Temps nettoyée',
    'data/ressources_clean.csv'            : 'Table Ressources nettoyée',
    'data/dossiers_enrichis.csv'           : 'Table jointe Dossiers + agrégats Temps',
    'reports/tableau_anomalies.csv'        : 'Tableau de synthèse des anomalies',
    'reports/bilan_completude.csv'         : 'Taux de complétude des colonnes clés',
    'reports/taux_activation_services.csv' : 'Taux d activation des services TOP',
    'reports/temps_par_dossier.csv'        : 'Temps total de traitement par dossier',
    'reports/indicateurs_activite.csv'     : 'Indicateurs clés — Activité',
    'reports/indicateurs_rh.csv'           : 'Indicateurs clés — Ressources humaines',
}

print('=== Fichiers produits par le pipeline ===\n')
for path, desc in fichiers.items():
    p = PROJECT_ROOT / path
    if p.exists():
        taille = p.stat().st_size / 1024
        print(f'  [OK]  {path:<45} | {taille:>8.1f} Ko | {desc}')
    else:
        print(f'  [--]  {path:<45} | non trouvé   | {desc}')

print('\n=== Graphiques (reports/figures/) ===\n')
pngs = sorted(FIGURES_DIR.glob('*.png'))
if pngs:
    for png in pngs:
        taille = png.stat().st_size / 1024
        print(f'  [OK]  figures/{png.name:<42} | {taille:>8.1f} Ko')
else:
    print('  Aucun graphique trouvé — relancer le notebook pour les générer.')

### 13.2 Dashboard interactif

Un dashboard Streamlit est disponible dans `mise_en_production/app.py`. Il permet d'explorer les données nettoyées de façon interactive, avec des filtres par année, client et outil.

**Pour le lancer** (depuis la racine du projet) :
```bash
streamlit run mise_en_production/app.py
```

Le dashboard charge directement les fichiers nettoyés (`data/dossiers_clean.csv`, etc.) — il faut donc avoir exécuté ce notebook au préalable pour que les données soient disponibles.

### 13.3 Rejouer le pipeline sur de nouvelles données

Pour mettre à jour les résultats avec un nouveau fichier source :

1. Déposer le nouveau CSV dans `C:/Users/Mame Diarra NDIAYE/Downloads/`
2. Mettre à jour le nom du fichier dans la cellule **Configuration** (Section 1)
3. Exécuter toutes les cellules : `Kernel → Restart & Run All`

Les fichiers nettoyés dans `data/` seront automatiquement écrasés et le dashboard se mettra à jour au prochain lancement.

### 13.4 Pistes d'analyses complémentaires

Ces analyses peuvent être développées dans un notebook dédié :

| Axe | Question | Outil suggéré |
|---|---|---|
| Prédiction | Peut-on prédire la durée de traitement d'un dossier ? | Régression (scikit-learn) |
| Segmentation | Existe-t-il des profils types de dossiers ? | Clustering (K-Means) |
| Performance agents | Quels agents traitent le plus vite sans sacrifier la qualité ? | Analyse comparative |
| Saisonnalité | Y a-t-il des patterns hebdomadaires en plus des patterns mensuels ? | Décomposition temporelle |
| Coût estimé | Quel est le coût moyen par type de sinistre ? | Simulation financière |

---
*Pipeline réalisé dans le cadre du Projet Data — Assurance Assistance 2021/2022*